<a href="https://colab.research.google.com/github/flg37/Beng-Dissertation/blob/main/Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
###
## SOH Labels
###

# ==============================================================================
# 0. ENVIRONMENT INITIALIZATION AND DEPENDENCIES
# ==============================================================================
from google.colab import drive
import pandas as pd
import numpy as np
import time
import multiprocessing
import os
import glob
from scipy.interpolate import interp1d
from joblib import Parallel, delayed
import gc

# Mount Google Drive to access the longitudinal vehicle telemetry dataset.
drive.mount('/content/drive')

# ==============================================================================
# 1. DIRECTORY AND PATH CONFIGURATION
# ==============================================================================
DRIVE_PREFIX = "/content/drive/MyDrive/Beng_Dissertation"
DRIVE_PREFIX2 = "/content/drive/MyDrive"
DATA_DIR = DRIVE_PREFIX  # Directory containing distributed per-vehicle CSVs
OCV_PATH = f"{DRIVE_PREFIX}/ocv_reference_curve2.csv"  # Empirical Open-Circuit Voltage reference
OUTPUT_PATH = f"{DRIVE_PREFIX}/session_level_soh_physics_final7.csv"  # Target path for final longitudinal output

# ==============================================================================
# 2. EMPIRICAL OCV-SOC MAPPING FUNCTION
# ==============================================================================
print("Loading OCV-SOC reference mapping...")

ref_df = pd.read_csv(OCV_PATH)

# Linear interpolation mapping cell-level OCV (V) to SOC (%).
# Extrapolation permitted for voltages marginally outside the characterized range.
ocv_mapper = interp1d(
    ref_df['ocv_cell'].values,
    ref_df['soc'].values,
    kind='linear',
    bounds_error=False,
    fill_value="extrapolate"
)

# ==============================================================================
# 3. PHYSICS-INFORMED STATE OF HEALTH (SOH) ESTIMATION ALGORITHM
# ==============================================================================
def calculate_vehicle_soh(df, vehicle_id, ca_rated=155.0):
    """
    Computes session-level SOH using Ah-throughput integration (Coulomb counting)
    and Ohmic drop compensation to estimate physical SOC.
    """
    sessions = []

    # Pre-processing: temporal sorting and deduplication.
    df = df.sort_values("terminaltime").drop_duplicates("terminaltime").reset_index(drop=True)

    # Isolate active charging states.
    df = df[df["chargestatus"] == 1].copy()
    if len(df) < 50:
        return sessions

    # Standardize timestamps to datetime objects.
    if not np.issubdtype(df["terminaltime"].dtype, np.datetime64):
        try:
            df["terminaltime"] = pd.to_datetime(df["terminaltime"], unit="s", errors="coerce")
        except Exception:
            df["terminaltime"] = pd.to_datetime(df["terminaltime"], errors="coerce")

    # Filter critical missing telemetry vectors.
    df = df.dropna(subset=["terminaltime", "soc", "totalcurrent", "maxtemperaturevalue", "totalvoltage"])
    df["soc_frac"] = df["soc"] / 100.0

    # Segment into discrete charging sessions using a 30-minute gap threshold.
    t_sec = df["terminaltime"].values.astype("datetime64[s]").astype(np.int64)
    dt_sec = np.insert(np.diff(t_sec), 0, 0)
    df["session_id"] = (dt_sec > 1800).cumsum()

    # Iterate over discrete charging sessions.
    for sid, g in df.groupby("session_id"):
        avg_temp = g["maxtemperaturevalue"].mean()

        # Constrain to nominal operating temperatures (15-35 C).
        if avg_temp < 15 or avg_temp > 35:
            continue

        # ROI: bulk constant-current phase (30%-90% SOC).
        g_linear = g[(g["soc_frac"] >= 0.30) & (g["soc_frac"] <= 0.90)].copy()
        if len(g_linear) < 100:
            continue

        t_start = g_linear["terminaltime"].iloc[0]
        t_end = g_linear["terminaltime"].iloc[-1]
        duration_min = (t_end - t_start).total_seconds() / 60.0

        N_s = 96  # Number of series-connected cells in the pack.

        # ------------------------------------------------------------------
        # DYNAMIC INTERNAL RESISTANCE (R_PACK) ESTIMATION via dV/dI
        # ------------------------------------------------------------------
        v_seg = g_linear["totalvoltage"].values.astype(float)
        i_seg = g_linear["totalcurrent"].values.astype(float)

        di = np.diff(i_seg)
        dv = np.diff(v_seg)

        # Identify substantial current steps to approximate local DC resistance.
        valid_ri = np.abs(di) > 1.0
        if valid_ri.sum() > 5:
            r_estimates = -dv[valid_ri] / di[valid_ri]
            # Constrain to plausible bounds (0.02 - 0.5 Ohms).
            r_estimates = r_estimates[(r_estimates > 0.02) & (r_estimates < 0.5)]
            R_pack = float(np.median(r_estimates)) if len(r_estimates) > 3 else 0.08
        else:
            R_pack = 0.08  # Default ESR fallback.

        # ------------------------------------------------------------------
        # OHMIC DROP COMPENSATION & PHYSICAL SOC ESTIMATION
        # ------------------------------------------------------------------
        v_start_pack = float(g_linear["totalvoltage"].iloc[0])
        i_start_pack = float(g_linear["totalcurrent"].iloc[0])
        ocv_pack_est = v_start_pack - (i_start_pack * R_pack)

        # Normalize to cell-level resting voltage.
        ocv_cell_est = ocv_pack_est / N_s

        # Map to physical SOC via the empirical reference curve.
        try:
            ocv_cell_est = np.clip(ocv_cell_est, 3.0, 4.25)
            soc_start_physical = float(ocv_mapper(ocv_cell_est)) / 100.0
        except Exception:
            continue

        soc_end_bms = g_linear["soc_frac"].iloc[-1]
        delta_soc = soc_end_bms - soc_start_physical

        # Require a sufficiently wide SOC window for robust integration.
        if delta_soc < 0.40 or duration_min < 45.0:
            continue

        # Reject non-monotonic SOC reports during charging.
        if np.any(np.diff(g_linear["soc_frac"]) < -0.01):
            continue

        seg_t_sec = g_linear["terminaltime"].values.astype("datetime64[s]").astype(np.int64)
        seg_curr = np.abs(g_linear["totalcurrent"].values.astype(float))

        # Reject trickle-charging profiles.
        if np.mean(seg_curr) < 8.0:
            continue

        # Reject sessions with significant data gaps (>15 minutes).
        dt_hours = np.diff(seg_t_sec) / 3600.0
        if np.any(dt_hours > (900.0 / 3600.0)):
            continue

        i_avg = (seg_curr[:-1] + seg_curr[1:]) / 2.0

        # ------------------------------------------------------------------
        # AH-THROUGHPUT INTEGRATION (COULOMB COUNTING)
        # ------------------------------------------------------------------
        integrated_ah = np.sum(i_avg * dt_hours)
        if integrated_ah <= 0:
            continue

        # Coulombic efficiency constant.
        eta_c = 0.995
        integrated_ah_true = integrated_ah * eta_c

        # C_raw = integral(I dt) / delta_SOC
        capacity_ah_raw = integrated_ah_true / delta_soc

        # ------------------------------------------------------------------
        # THERMAL NORMALIZATION & SOH DERIVATION
        # ------------------------------------------------------------------
        # Linear temperature coefficient (alpha) to normalize to 25 C.
        alpha = 0.0025
        capacity_ah_norm = capacity_ah_raw / (1 + alpha * (avg_temp - 25))

        # SOH relative to the rated End-of-Line benchmark.
        soh = capacity_ah_norm / ca_rated

        # Filter out-of-bounds capacities indicative of sensor failure.
        if soh > 1.10 or soh < 0.65:
            continue

        sessions.append({
            'vehicle_id': vehicle_id,
            'session_id': int(sid),
            'start_time': t_start,
            'end_time': t_end,
            'duration_min': duration_min,
            'avg_temp': avg_temp,
            'r_pack_estimated': R_pack,
            'soc_start_bms': g_linear["soc_frac"].iloc[0],
            'soc_start_physical': soc_start_physical,
            'soc_end': soc_end_bms,
            'delta_soc': delta_soc,
            'integrated_ah': integrated_ah_true,
            'estimated_capacity_ah': capacity_ah_norm,
            'soh': soh,
            'q_proxy': float(capacity_ah_norm)
        })

    # ==========================================================================
    # INTRA-VEHICLE STATISTICAL REJECTION & SMOOTHING
    # ==========================================================================
    if len(sessions) >= 3:
        df_sess = pd.DataFrame(sessions).sort_values('start_time')

        # IQR filtering to reject statistical outliers.
        Q1, Q3 = df_sess['soh'].quantile(0.25), df_sess['soh'].quantile(0.75)
        IQR = Q3 - Q1
        df_sess = df_sess[
            (df_sess['soh'] >= Q1 - 1.5 * IQR) &
            (df_sess['soh'] <= Q3 + 1.5 * IQR)
        ]

        # Rolling median filter to validate against the vehicle's degradation trend.
        if len(df_sess) >= 3:
            trendline = df_sess['soh'].rolling(window=10, min_periods=3, center=True).median()
            valid_mask = (np.abs(df_sess['soh'] - trendline) / trendline) <= 0.04
            df_sess = df_sess[valid_mask]

        return df_sess.to_dict('records')

    return sessions

# ==============================================================================
# 4. DATA PIPELINE: SINGLE-VEHICLE WORKER
# ==============================================================================
def process_single_file_drive(file_path, ca_rated):
    """
    Ingests a single vehicle's telemetry, runs the session-level SOH
    algorithm, and frees local memory.
    """
    try:
        filename = os.path.basename(file_path)

        # Extract VIN identifier.
        veh_id = filename.replace('.csv', '').replace('vin', '')

        # Load only required features.
        cols_to_use = [
            'terminaltime', 'chargestatus', 'soc',
            'totalvoltage', 'totalcurrent', 'totalodometer',
            'maxtemperaturevalue'
        ]
        df = pd.read_csv(file_path, usecols=cols_to_use, low_memory=False)

        sessions = calculate_vehicle_soh(df, veh_id, ca_rated)

        del df
        gc.collect()
        return sessions

    except Exception:
        # Failsafe for malformed files.
        return []

# ==============================================================================
# 5. DISTRIBUTED BATCH PROCESSING PIPELINE
# ==============================================================================
def process_all_vehicles_drive(data_dir, output_path, ca_rated=155.0):
    """
    Orchestrates parallel processing across CPU cores, runs the SOH method
    over the full fleet, then applies macro-level longitudinal post-processing
    (monotonic degradation enforcement).
    """
    print("\n" + "=" * 70)
    print(" LONGITUDINAL BATTERY SOH ESTIMATION (PHYSICS-INFORMED METHODOLOGY)")
    print("=" * 70)

    # Recursively find all vehicle CSVs.
    all_files = sorted(glob.glob(f"{data_dir}/**/*.csv", recursive=True))
    print(f"Discovered {len(all_files)} telemetry sets.")

    num_cores = multiprocessing.cpu_count()
    print(f"Allocating {num_cores} virtual cores for concurrent processing.")

    all_sessions = []
    start_time = time.time()
    chunk_size = 25

    # Chunk workloads to manage serialization overhead.
    file_chunks = [all_files[i:i + chunk_size] for i in range(0, len(all_files), chunk_size)]
    vehicles_with_data = 0

    with Parallel(n_jobs=num_cores, backend="loky", timeout=800) as parallel:
        for chunk_idx, chunk in enumerate(file_chunks):
            try:
                results = parallel(
                    delayed(process_single_file_drive)(path, ca_rated)
                    for path in chunk
                )
            except Exception as e:
                print(f"Node failure or timeout at batch {chunk_idx}: {e}")
                continue

            for vehicle_sessions in results:
                if len(vehicle_sessions) > 0:
                    all_sessions.extend(vehicle_sessions)
                    vehicles_with_data += 1

            # Checkpoint intermediate state.
            if len(all_sessions) > 0:
                checkpoint_df = pd.DataFrame(all_sessions)
                checkpoint_path = output_path.replace('.csv', '_checkpoint.csv')
                checkpoint_df.to_csv(checkpoint_path, index=False)

            # Progress reporting.
            processed_so_far = min((chunk_idx + 1) * chunk_size, len(all_files))
            elapsed = time.time() - start_time
            rate = processed_so_far / (elapsed / 60.0) if elapsed > 0 else 0.0
            remaining = (len(all_files) - processed_so_far) / rate if rate > 0 else 0.0

            print(
                f"[{processed_so_far}/{len(all_files)}] | "
                f"Vehicles yielding active series: {vehicles_with_data} | "
                f"Global validated epochs: {len(all_sessions)} | "
                f"{rate:.1f} nodes/min | ETA: {remaining:.1f} min"
            )

    # --------------------------------------------------------------------------
    # MACRO-LEVEL POST-PROCESSING: Trajectory Normalization & Heuristics
    # --------------------------------------------------------------------------
    if len(all_sessions) > 0:
        results_df = pd.DataFrame(all_sessions)

        if not results_df.empty:
            # Strict chronological ordering per VIN.
            results_df["start_time"] = pd.to_datetime(results_df["start_time"])
            results_df = results_df.sort_values(["vehicle_id", "start_time"]).reset_index(drop=True)

            def add_relative_soh(group):
                """
                Relative SOH: normalizes observations against the vehicle's
                early-life capacity baseline.
                """
                group = group.sort_values("start_time").copy()

                # Baseline from early-life sessions (75th percentile of first 10).
                baseline_pool = group.head(min(10, len(group)))
                baseline_q = baseline_pool["q_proxy"].quantile(0.75)

                if pd.isna(baseline_q) or baseline_q <= 0:
                    group["soh_rel"] = np.nan
                    group["soh_rel_final"] = np.nan
                    return group

                group["soh_rel"] = group["q_proxy"] / baseline_q

                # Enforce monotonic degradation: capacity cannot regenerate.
                group["soh_rel_final"] = np.minimum.accumulate(group["soh_rel"])
                return group

            results_df = results_df.groupby("vehicle_id", group_keys=False).apply(add_relative_soh)
            results_df["start_time"] = pd.to_datetime(results_df["start_time"])

            def add_time_axis(group):
                """
                Adds a continuous time axis for life-cycle modeling and
                degradation-rate extraction.
                """
                group = group.sort_values("start_time").copy()
                t0 = group["start_time"].iloc[0]
                group["years_since_start"] = (group["start_time"] - t0).dt.total_seconds() / (365.25 * 24 * 3600)
                return group

            results_df = results_df.groupby("vehicle_id", group_keys=False).apply(add_time_axis)

            # Final boundary constraints to exclude impossible states.
            results_df = results_df[
                (results_df["soh_rel_final"] >= 0.6) &
                (results_df["soh_rel_final"] <= 1.10)
            ].copy()

            # Persist to storage.
            results_df.to_csv(output_path, index=False)
    else:
        results_df = pd.DataFrame()

    total_min = (time.time() - start_time) / 60.0
    print(f"\nCOMPUTATION TERMINATED: Elapsed duration {total_min:.1f} minutes")
    print(f"Matrix persisted to target schema: {output_path}")

    return results_df

# ==============================================================================
# 6. SCRIPT ENTRY POINT & EXECUTION CONTEXT
# ==============================================================================
CA_RATED = 155.0  # Nominal pack capacity (Ah) used for absolute normalization

print("Initiating global physics-informed SOH extraction pipeline from Cloud Storage...")
results = process_all_vehicles_drive(DATA_DIR, OUTPUT_PATH, ca_rated=CA_RATED)

if not results.empty:
    print("\nArray verification sample:\n", results.head(10).to_string())

print("\nRoutine finalized successfully.")

Mounted at /content/drive
Loading OCV-SOC reference mapping...
Initiating global physics-informed SOH extraction pipeline from Cloud Storage...

 LONGITUDINAL BATTERY SOH ESTIMATION (PHYSICS-INFORMED METHODOLOGY)
Discovered 39 telemetry sets.
Allocating 2 virtual cores for concurrent processing.


KeyboardInterrupt: 

In [ ]:
###
## Feature Extraction
##

# ==============================================================================
# 1. ENVIRONMENT INITIALIZATION AND DEPENDENCIES
# ==============================================================================
import pandas as pd
import numpy as np
import os
import time
import pickle
import scipy.ndimage
import sys
import glob
import gc
from joblib import Parallel, delayed
from tqdm.notebook import tqdm

print("========================================================================")
print(" STAGE 2: DISTRIBUTED MULTI-MODAL FEATURE EXTRACTION (PRODUCTION PIPELINE)")
print("========================================================================")

# ------------------------------------------------------------------------------
# DIRECTORY & I/O CONFIGURATION
# ------------------------------------------------------------------------------
DRIVE_PREFIX = "/content/drive/MyDrive/Beng_Dissertation"
DRIVE_PREFIX2 = "/content/drive/MyDrive"

# Input: Ground-truth SOH evaluation epochs generated from the physics-based model.
LABELS_PATH = f"{DRIVE_PREFIX}/session_level_soh_natural_final2.csv"
DATA_DIR = DRIVE_PREFIX

# Output: Serialized multi-modal feature tensors (ready for PyTorch DataLoaders).
OUTPUT_PATH = f"{DRIVE_PREFIX}/multimodal_features_synced6.pkl"

# Translation table for parsing the cell-voltage strings in the telemetry arrays.
CLEAN_TRANS = str.maketrans("~", ",", "[]'\"\n ")

# ==============================================================================
# 2. TEMPORAL SYNCHRONIZATION MATRIX CONSTRUCTION
# ==============================================================================
print(" Ingesting Ground-Truth SOH Labels to construct Temporal Anchors...")
try:
    with open(LABELS_PATH, "rb") as f:
        soh_labels = pd.read_csv(f)

    MASTER_TIME_DICT = {}
    vids = soh_labels['vehicle_id_split'].astype(str).values
    times = pd.to_datetime(soh_labels['start_time']).values
    sids = soh_labels['session_id'].astype(int).values

    # O(1) lookup dictionary mapping vehicle identifiers to their validated
    # charging session timestamps (temporal anchors).
    for i in range(len(vids)):
        vid = vids[i]
        if vid not in MASTER_TIME_DICT:
            MASTER_TIME_DICT[vid] = []
        MASTER_TIME_DICT[vid].append((pd.Timestamp(times[i]), sids[i]))

    # Deduplicate core physical vehicle IDs from synthetic split identities.
    successful_vids = set(str(vid).split('_gen')[0] for vid in MASTER_TIME_DICT.keys())
    print(f" Validated {len(soh_labels)} SOH labels across {len(successful_vids)} physical assets.")

except Exception as e:
    print(f" Fatal Error: SOH label ingestion failed: {e}")
    sys.exit()

# Traverse the file system and keep only telemetry logs for validated vehicles.
all_files = glob.glob(f"{DATA_DIR}/**/*.csv", recursive=True)
target_files = [
    f for f in all_files
    if f.split('/')[-1].replace('.csv', '').replace('vin', '') in successful_vids
]
print(f" Pipeline filtered to {len(target_files)} target telemetry matrices.\n")

# ==============================================================================
# 3. CORE MULTI-MODAL FEATURE ENGINEERING ALGORITHM
# ==============================================================================
def extract_optimized_session(df, vehicle_id, local_sessions):
    """
    Transforms raw chronological telemetry into structured, multi-modal feature
    tensors synchronized with ground-truth SOH labels.

    Generates three distinct data modalities:
    1. point_features: Scalar macro-statistics (for MLP heads).
    2. voltage_map_2d: 96x96 spatio-temporal cell voltage projection (for 2D CNNs / Vision Mamba).
    3. capacity_seq_1d: Sequential Ah-throughput and thermal profiles (for RNNs / 1D SSMs).
    """
    features = []

    # Chronological enforcement and localized deduplication.
    df = df.sort_values("terminaltime").drop_duplicates("terminaltime").reset_index(drop=True)

    # Cast raw temporal vectors to datetime.
    if not np.issubdtype(df["terminaltime"].dtype, np.datetime64):
        try:
            df["terminaltime"] = pd.to_datetime(df["terminaltime"], unit="s", errors="coerce")
        except Exception:
            df["terminaltime"] = pd.to_datetime(df["terminaltime"], errors="coerce")

    # Filter critical state vectors.
    df = df.dropna(subset=["terminaltime", "soc", "totalcurrent", "maxtemperaturevalue", "totalvoltage"])
    if len(df) < 15:
        return []

    # Segment continuous telemetry into discrete epochs (30-minute threshold).
    t_sec = df["terminaltime"].values.astype("datetime64[s]").astype(np.int64)
    dt_sec = np.diff(t_sec)
    dt_sec = np.insert(dt_sec, 0, 0)
    df["internal_session_id"] = (dt_sec > 1800).cumsum()

    CLEAN_TRANS = str.maketrans("~", ",", "[]'\"\n ")

    for internal_id, session in df.groupby('internal_session_id'):
        session_start_time = session['terminaltime'].iloc[0]
        assigned_vid_split = None
        assigned_session_id = None

        # Cross-reference the epoch with the ground-truth temporal anchors.
        # Uses a +/- 30 minute (1800s) synchronization tolerance window.
        for v_split, vt, sid in local_sessions:
            if abs((session_start_time - vt).total_seconds()) < 1800:
                assigned_vid_split = v_split
                assigned_session_id = sid
                break

        # Discard epochs lacking a verified corresponding SOH label.
        if not assigned_vid_split:
            continue
        if len(session) < 15:
            continue

        try:
            session["soc_frac"] = session["soc"] / 100.0

            # Isolate the stable pseudo-linear region of the charging curve.
            crit = session[(session["soc_frac"] >= 0.20) & (session["soc_frac"] <= 0.85)].copy()
            if len(crit) < 5:
                crit = session.copy()  # Fallback if the constrained window is too narrow.

            # ------------------------------------------------------------------
            # CELL-LEVEL VOLTAGE MATRIX DESERIALIZATION
            # ------------------------------------------------------------------
            bv_series = crit['batteryvoltage'].dropna().astype(str)
            raw_cells = [np.fromstring(x.translate(CLEAN_TRANS), sep=',') for x in bv_series]

            # Enforce strict spatial dimensionality (96 series-connected cells).
            raw_cells = [arr for arr in raw_cells if len(arr) == 96]
            if len(raw_cells) < 5:
                continue

            # Spatio-temporal cell voltage matrix (Time Steps x 96 Cells).
            cell_matrix = np.vstack(raw_cells).astype(np.float32)

            # Intra-pack voltage dispersion (mV) as a localized variance metric.
            dv_series = np.ptp(cell_matrix, axis=1) * 1000.0

            # ------------------------------------------------------------------
            # AH-THROUGHPUT INTEGRATION (COULOMB COUNTING)
            # ------------------------------------------------------------------
            times_val = crit['terminaltime'].values.astype('datetime64[s]').astype(np.int64)
            dt_step = np.diff(times_val, prepend=times_val[0]) / 3600.0
            q_series = np.cumsum(crit['totalcurrent'].abs().values * dt_step)

            # Monotonic reference axis (max cell voltage, or mean cell voltage as fallback).
            if 'maxvoltagebattery' in crit.columns and crit['maxvoltagebattery'].notna().any():
                v_source = crit['maxvoltagebattery'].ffill().bfill().values
            else:
                v_source = (crit['totalvoltage'] / 96.0).values

            # Guarantee strict monotonic progression for numerical interpolation.
            order = np.argsort(v_source)
            v_source_sorted = v_source[order]
            v_unique, idx_unique = np.unique(v_source_sorted, return_index=True)

            # ------------------------------------------------------------------
            # MODALITY 1: 2D SPATIO-TEMPORAL VOLTAGE PROJECTION (96x96)
            # ------------------------------------------------------------------
            # Project time-domain cell voltages onto a standardized 96-bin voltage
            # grid (3.90V to 4.05V). This generates a fixed-dimension 2D tensor
            # suitable for Vision Mamba or ResNet ingestion.
            v_grid = np.linspace(3.900, 4.050, 96)
            voltage_map = np.zeros((96, 96), dtype=np.float32)
            cell_matrix_sorted = cell_matrix[order][idx_unique]

            for c in range(96):
                voltage_map[:, c] = np.interp(v_grid, v_unique, cell_matrix_sorted[:, c])

            # ------------------------------------------------------------------
            # MODALITY 2: 1D SEQUENTIAL PROFILES
            # ------------------------------------------------------------------
            t_series = crit['maxtemperaturevalue'].values
            q_unique = q_series[order][idx_unique]
            t_unique = t_series[order][idx_unique]

            # Interpolate scalar sequences onto a higher-resolution 151-point grid.
            v_target = np.linspace(3.900, 4.050, 151)
            seq_q = np.interp(v_target, v_unique, q_unique).astype(np.float32)
            seq_t = np.interp(v_target, v_unique, t_unique).astype(np.float32)

            # Zero-center the capacity vector to capture purely relative charge acceptance.
            seq_q = seq_q - seq_q[0]

            # ------------------------------------------------------------------
            # MODALITY 3: MACROSCOPIC POINT FEATURES
            # ------------------------------------------------------------------
            # Statistical moments (mean, variance, PtP) representing bulk session dynamics.
            p_feat = np.array([
                np.nanmax(df['totalodometer'].values),
                session['maxtemperaturevalue'].mean(),
                np.mean(seq_q), np.median(seq_q), np.std(seq_q), np.ptp(seq_q),
                np.mean(dv_series), np.median(dv_series), np.std(dv_series), np.ptp(dv_series),
                session['totalcurrent'].max(), session['totalcurrent'].mean(),
                session['totalvoltage'].max(),
                session['mintemperaturevalue'].min(), session['mintemperaturevalue'].mean()
            ], dtype=np.float32)

            # Package synchronized modalities for serialization.
            features.append({
                'vehicle_id': assigned_vid_split,
                'session_id': int(assigned_session_id),
                'point_features': p_feat,
                'voltage_map_2d': voltage_map,
                'capacity_seq_1d': seq_q,
                'temperature_seq_1d': seq_t
            })

        except Exception:
            continue

    return features

# ==============================================================================
# 4. DISTRIBUTED WORKER NODE FUNCTION
# ==============================================================================
def worker_task(f, master_dict):
    """
    Independent execution node: loads a vehicle's telemetry, matches it against
    the global temporal dictionary, and extracts features.
    """
    try:
        vid = f.split('/')[-1].replace('.csv', '').replace('vin', '')

        # Isolate temporal anchors relevant only to this worker's vehicle.
        local_sessions = []
        for v_split, times_list in master_dict.items():
            if str(v_split).startswith(vid):
                for vt, sid in times_list:
                    local_sessions.append((v_split, vt, sid))

        if not local_sessions:
            return []

        # Load only the required columns to limit memory use.
        cols_to_load = ['terminaltime', 'chargestatus', 'totalvoltage',
                        'totalcurrent', 'maxtemperaturevalue',
                        'mintemperaturevalue', 'batteryvoltage',
                        'totalodometer', 'soc', 'maxvoltagebattery']

        with open(f, "rb") as file_obj:
            df = pd.read_csv(file_obj, usecols=cols_to_load,
                             dtype={'batteryvoltage': str}, low_memory=False)

        # Restrict to active charging epochs.
        df = df[df['chargestatus'] == 1]
        if df.empty:
            return []

        res = extract_optimized_session(df, vid, local_sessions)

        # Explicit garbage collection in the Loky subprocess.
        del df
        gc.collect()

        return res

    except Exception as e:
        print(f" I/O Error during ingestion of {f}: {e}")
        return []

# ==============================================================================
# 5. ASYNCHRONOUS BATCH PROCESSING PIPELINE
# ==============================================================================
BATCH_SIZE = 5
final_features = []

print(" Initiating Cloud-Scale Parallel Extraction (Loky Backend)...")
start_time = time.time()

# Chunk files to balance serialization overhead and worker saturation.
chunks = [target_files[i:i + BATCH_SIZE] for i in range(0, len(target_files), BATCH_SIZE)]

# Multi-core orchestration using all available vCPUs (n_jobs=-1).
for i, batch in enumerate(tqdm(chunks, desc="Processing Synchronized Batches")):
    results = Parallel(n_jobs=-1, backend='loky')(
        delayed(worker_task)(f, MASTER_TIME_DICT) for f in batch
    )

    # Aggregate extracted tensors into the global state.
    for res in results:
        if res:
            final_features.extend(res)

    del results
    gc.collect()

# ==============================================================================
# 6. TENSOR SERIALIZATION & PERSISTENCE
# ==============================================================================
if len(final_features) > 0:
    print(f"\n Persisting {len(final_features)} multi-modal feature blocks to Cloud Storage...")

    # Pickle preserves the native NumPy FP32 structure.
    with open(OUTPUT_PATH, 'wb') as f:
        pickle.dump(final_features, f)

    print(f" PROCESS COMPLETE: Elapsed execution time {(time.time() - start_time)/60:.1f} minutes.")
    print(f" Target Tensor Output: {OUTPUT_PATH}")
else:
    print("\n WARNING: Extraction yielded zero valid tensors. Verify synchronization tolerances.")